# Borealis-27b at full bf16 — NorSumm + NorQuAD + QGEval

Runs Borealis-27b variants at their **native bf16 precision — nothing quantized** — over **three** benchmarks:

| Benchmark | Task | Metric | Size | Models per run |
|---|---|---|---|---|
| **NorSumm** | summarization | ROUGE-1/2/L | 33 articles | one — set `MODEL_ID` in 5b |
| **NorQuAD** | extractive QA | Exact Match / F1 | 300 questions | one — set `MODEL_ID` in 5b |
| **QGEval** | answer-conditioned question generation | 7 dimensions (Fluency, Clarity, Conciseness, Relevance, Consistency, Answerability, Answer Consistency) | 200 passages | **both variants automatically** — cell 9 loops over them |

## Which model(s)

Two Borealis variants exist, and they are **not** the same thing:

| Model | What it is | bf16 size |
|---|---|---|
| `NbAiLab/borealis-27b` | instruction-tuned **full release** | 51.1 GiB |
| `NbAiLab/borealis-27b-instruct-preview` | earlier **preview** — its own card calls it *"an experiment ... pre-release quality"*, tuned from `google/gemma-3-27b-it` | 53.7 GiB |

The preview is **not** a newer or better model. Output filenames carry the model name, so the two never overwrite each other.

**NorSumm and NorQuAD QA** are per-`MODEL_ID`: set it in cell 5b, run cells 7 and/or 8, download, then change `MODEL_ID` and repeat for the other variant.

**QGEval (cell 9)** is different — it ignores whatever `MODEL_ID` you set in 5b and instead loops over **both** variants itself, one full model load at a time with memory freed in between. Run it once and you get both.

## Before you start

**Runtime → Change runtime type → `A100 GPU` + `High-RAM`.**

On **Colab Pro+**, also enable **background execution**. All three benchmarks off one model load takes several hours; cell 9 alone (two full models, sequentially) also takes hours.

Neither model fits on Colab's 40 GB A100, so accelerate splits it across GPU and CPU RAM. Everything stays bf16; the offloaded layers just make generation slow.

**For Benchmarks A/B, the model is loaded once and reused** — do not re-run the load cell between them. **Cell 9 (QGEval) is independent of that** — it manages its own model loads and frees each before the next, so it can run before, after, or without Benchmarks A/B ever running in this session.

## 1. Check what card you were assigned

Run this first. If it says anything other than A100, use **Runtime → Disconnect and delete runtime** and try again — Colab assigns cards from a pool.

In [ ]:
import torch, psutil

WEIGHTS = 53.7  # GiB — larger of the two variants; cell 5b prints the exact figure

if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime > Change runtime type > A100 GPU + High-RAM.')

name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
ram = psutil.virtual_memory().total / 1024**3
gpu_budget = vram - 6
spill = max(0.0, WEIGHTS - gpu_budget)

print(f'GPU: {name} ({vram:.1f} GiB)')
print(f'RAM: {ram:.1f} GiB')
print(f'Weights: {WEIGHTS} GiB bf16\n')

if vram < 30:
    print(f'STOP. {name} is too small. Disconnect and delete the runtime, then retry for an A100.')
elif spill == 0:
    print('Excellent — fully GPU-resident. Expect ~10-20 minutes for all 33 articles.')
elif spill > ram - 8:
    print(f'STOP. {spill:.1f} GiB must spill to RAM but only {ram-8:.1f} GiB is usable.')
    print('Switch to a High-RAM runtime.')
else:
    print(f'OK: {gpu_budget:.1f} GiB on GPU, {spill:.1f} GiB offloaded to RAM.')
    print('Expect ~1-3 tok/s, roughly 1.5-4 hours for all 33 articles.')
    if ram < 60:
        print('\nNOTE: this looks like standard-RAM. High-RAM is strongly recommended.')

## 2. Install dependencies

`bitsandbytes` is deliberately **not** installed — this run is unquantized.

In [ ]:
!pip install -q -U transformers accelerate huggingface_hub pandas pyarrow
import transformers, accelerate
print('transformers', transformers.__version__, '| accelerate', accelerate.__version__)

## 3. Mount Drive (recommended)

The script checkpoints after **every article**. Writing those checkpoints to Drive means a disconnect costs one article instead of the whole run. Skip this cell and it falls back to local storage, which is wiped when the runtime dies.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# One subdirectory per benchmark — all three write the same filename for a given model.
!mkdir -p /content/drive/MyDrive/borealis_bench/norsumm /content/drive/MyDrive/borealis_bench/qa /content/drive/MyDrive/borealis_bench/qgeval

## 4. Get the benchmark code

Public repo, so no token needed.

In [ ]:
!git clone -q --branch claude/norquad-norsumm-benchmarks-9qhyke \
    https://github.com/rymarinelli/embedding.git /content/embedding
%cd /content/embedding/benchmarks/scripts
!ls generate_summaries_borealis_bf16_colab.py generate_qa_answers_borealis_bf16_colab.py generate_questions_qgeval_borealis_colab.py
!ls ../data/qgeval_sample.json

## 5. (Optional) Hugging Face login

`NbAiLab/borealis-27b` is not gated, so this is only useful for download rate limits. Skip it if you like.

In [ ]:
# from huggingface_hub import login
# login(token='hf_...')   # never commit this

## 5b. Choose the model (for Benchmarks A/B only)

Edit `MODEL_ID` here. Everything downstream in **Benchmarks A and B** — weight-size checks, memory budgets, output filenames and result labels — follows from it.

**Benchmark C (QGEval, cell 9) ignores this** — it runs both variants itself, regardless of what's set here.

In [ ]:
import generate_summaries_borealis_bf16_colab as run

# 'NbAiLab/borealis-27b'                  -> instruction-tuned full release (51.1 GiB)
# 'NbAiLab/borealis-27b-instruct-preview' -> earlier preview, pre-release quality (53.7 GiB)
run.MODEL_ID = 'NbAiLab/borealis-27b-instruct-preview'
run.RUN_LABEL, run.OUT_NAME, run.OUT_PATH = run.derive_paths(run.MODEL_ID)

print('model      :', run.MODEL_ID)
print('bf16 weights:', f'{run.model_weight_gib():.1f} GiB')
print('will score as:', run.RUN_LABEL)

## 6. Run

Downloads ~51 GiB, then generates. Watch the **first article's timing**: if it takes more than ~8 minutes, the offload is thrashing and the run won't finish in a sitting — stop and confirm you got High-RAM.

`REPETITION_PENALTY` is left at **1.0**, matching the original quantized run, so this isolates the effect of precision alone. Only change it for a separate second run.

In [ ]:
# Load once, keep the handles — the QA benchmark below reuses them.
run.preflight()
processor, model = run.load_model()

print('\nrepetition_penalty =', run.REPETITION_PENALTY, '| temperature =', run.TEMPERATURE)
print('checkpointing to  =', run.OUT_PATH)

## 7. Benchmark A — NorSumm (summarization)

33 articles, ROUGE-1/2/L. Watch the **first article's timing**: more than ~8 minutes means the offload is thrashing — stop and confirm you got High-RAM.

Checkpoints after every article, so a disconnect costs one article. Re-run this cell to resume.

In [ ]:
run.main(model=model, processor=processor)

## 8. Benchmark B — NorQuAD (extractive QA)

300 questions, Exact Match / token-F1 — the numbers in Table 2. **Reuses the model already loaded above**, so there is no second 51 GiB download.

Decoding is greedy with `max_new_tokens=64`, matching `generate_qa_answers_local_gguf.py` so the EM/F1 are comparable with the API models. Note this differs from the summarization run, which matched its own baseline at temperature 0.2.

300 short answers on offloaded weights still takes a while, but each is far shorter than a summary. Checkpoints every question.

In [ ]:
import generate_qa_answers_borealis_bf16_colab as qa

qa.main(model=model, processor=processor)

## 9. Benchmark C — QGEval, BOTH Borealis variants

Generates the 200 answer-conditioned QGEval questions with **both** `NbAiLab/borealis-27b` (full release) and `NbAiLab/borealis-27b-instruct-preview`, one after the other, in this one cell — you don't need to re-run the whole notebook per model like Benchmarks A/B require.

**This cell is self-contained.** It does not reuse whatever model Benchmarks A/B loaded above — it loads each Borealis variant fresh, generates its 200 questions, then explicitly frees that model (`del` + `gc.collect()` + `torch.cuda.empty_cache()`) before loading the next. That keeps peak memory to one model at a time, at the cost of two ~51-54 GiB downloads instead of one — the download is cached by Hugging Face after the first time, so a re-run in the same runtime is fast.

Each variant checkpoints to its own file per question, so a Colab disconnect mid-loop costs at most the passages not yet done on whichever variant was running, and re-running this cell resumes both from where they stopped.

Expect roughly the same per-question pace as Benchmark B (QA) — short generations (`max_new_tokens=64`) — so noticeably faster than Benchmark A's 500-token summaries, but doubled since it's two full models.

In [ ]:
import gc
import torch
import generate_summaries_borealis_bf16_colab as run
import generate_questions_qgeval_borealis_colab as qg

# Free anything Benchmarks A/B left loaded, so this starts from a clean
# memory baseline regardless of what ran earlier in the session.
for name in ('model', 'processor'):
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()

BOREALIS_VARIANTS = [
    'NbAiLab/borealis-27b',                   # instruction-tuned full release
    'NbAiLab/borealis-27b-instruct-preview',  # earlier preview, pre-release quality
]

for model_id in BOREALIS_VARIANTS:
    print('=' * 70)
    print('QGEval generation for:', model_id)
    print('=' * 70)
    run.MODEL_ID = model_id
    m, p = qg.main()   # loads its own model+processor for this variant
    del m, p
    gc.collect()
    torch.cuda.empty_cache()
    print(f'Freed {model_id} from memory.\n')

print('Both variants done. Files are in MyDrive/borealis_bench/qgeval/:')
print('  borealis-27b.json')
print('  borealis-27b-instruct-preview.json')

## 10. Download all result files

| File | Goes in | Then run |
|---|---|---|
| `<model>-bf16-full.json` (summaries) | `benchmarks/results/generated_summaries/` | `score_summaries.py` |
| `<model>-bf16-full.json` (QA) | `benchmarks/results/qa_answers/` | `score_qa.py` |
| `<model-name>.json` (QGEval) | `benchmarks/results/qgeval_questions/` | `score_qgeval_dimensions_multi.py` then `report_qgeval_multi.py` |

The QGEval file's name is just the bare model name (e.g. `borealis-27b-instruct-preview.json`), not `-bf16-full` — the multi-model QGEval scripts key on the plain model name across every generation path (OpenRouter, local GGUF, this notebook).

In [ ]:
from google.colab import files
import os, json

base = '/content/drive/MyDrive/borealis_bench'
for task in ('norsumm', 'qa', 'qgeval'):
    d = os.path.join(base, task)
    if not os.path.isdir(d):
        print(f'{task}: nothing yet'); continue
    for f in sorted(os.listdir(d)):
        if f.endswith('.json'):
            path = os.path.join(d, f)
            print(f'{task}: {f} ({len(json.load(open(path)))} records)')
            files.download(path)